# 2. Professional Competition

RSTT provides base tournament class to generate simple competition. However realistic dataset involves many tournaments organized in circuits. Think of sport regular season and playoffs with the addition of a cup and international tournaments to which team qualify based on previous results. 

## Competition
[Competition](https://rstt.readthedocs.io/en/latest/rstt.scheduler.tournament.html#rstt.scheduler.tournament.competition.Competition) is an abstract class from which inherit tournament format such as [Round-Robin]. It takes as input a unique name, a ranking for seeding purposes and a solver to assign score to each games. It output games and a final standing. The usage is straight forward and intuitif:

```python
# Play a bunch of game
cup = RoundRobin(name="Spring 2026", seeding=WorldRanking, LogSolver())
cup.registration(teams)
cup.run()

# Ouput
games = cup.games()
standing = cup.standing()
```

The outputs play a crucial role in our project. We need games to update the GPR. And we need standing to qualify teams to the different event of a LoL calendar.


#### Scope
For simplification we will focus on the top level of competition and assume:
- A set of region
- During a year each region plays three "split"
- A split consist of a 3 stages: main-stage, play-ins and play-offs
- After each splits, each region sends their best teams to an international season finals: [FirstStand](),[MSI]() and at the end of the year [Wordls]()
- Each season finals will consist of 3 stages: main-stage, play-ins and play-offs


#### Goals
- Build a custom Competition class that plays all three stages of Split/Season Finals in a single run() call.
- Build a year calendar that runs automaticaly every event in a year.


#### Materials
- scene.py provide enum class for Region / Split / Finals
- data/qualifications.json provides a description of how teams participate in events and under which conditions
- data/teams provide LoL pro teams names and their regions.

## 1. StagedEvent: Implementation of a Competition


When working on a Competition, you want to be able to instanciate it with seedings, a solver and a name (used as identifier). You also want to 'register' participants and 'run' it. The following code snippet illustrate how the magic happends. It is **not** the exact implementation, but gives the necessary information to understand the logic of the class.

```python
def run(self):
    self.start(): ->
        self.seeding = self.seeding.fit(self.participants)
        self._initialise() # Optional Override (TODO ?)
    self.play(): ->
        while not self.finished:
            games = self.generate_games() # Abstract Method (TODO 1)
            games = self.play_games(games)
            self.edit(games): ->
                self.played_matches.append(games)
                self._update() # Optional Override (TODO ?)
                self.__finished = self._end_of_Stage() # Aabstract Method (TODO 2)
    self.trophies(): ->
        self._standing() # Abstract Method (TODO 3)
        for player in self.participants:
            player.collect(self)
```

In the project/scheduler/StagedEvent.py file there is a StageEvent class for you to fill. 

#### TODO:

In project/stagedevent.py, edit the StagedEvent class:
- Implement all three abstract methods: generate_games() / _end_of_stage() / _standing()
- decide what you want to do with _initialise()/ _update()
- feel free to add your own functionalities

Make sure the next cell is exectuable

In [1]:
from rstt import Player, BTRanking, BetterWin
from rstt import RoundRobin, SingleEliminationBracket, DoubleEliminationBracket

from project import StagedEvent

# simulation parameters
teams = Player.create(nb=8)
gt = BTRanking(name='GroundTruth', players=teams)
solver = BetterWin()

# stage parameters
stages = [RoundRobin, SingleEliminationBracket, DoubleEliminationBracket]
names = ['PlayIns', 'MainStage', 'PlayOffs']

# try your implementation 
test = StagedEvent(name='test', seeding=gt, solver=solver, stages=stages, stage_names=names)
test.registration(teams)
test.run()

## Testing StagedEvent behaviours



#### Hint
If you are stuck with the dataset size, DoubleEliminationBracket implementation runs the upper bracket in its initialise methods. 
As a consequences the corresponding games are not return by generate_games(). You need a work arround to capture them.

In [2]:
# 1.a check all teams participated in the tournaments
assert set(test.participants()) == set(teams), "❌ problem with the teams participating in the event"
print("✅ all teams in event")

# 1.b check all teams participated in the first stage
assert set(test.stages[0].participants()) == set(teams), "❌ problem with the teams participating in the first stage"
print("✅ all teams in first stage")

# 2.a each stage have started
for stage in test.stages:
    assert stage.started(), f"❌ {stage.name()} has not yet started !!! Did you forget to call stage.start() ?"
print("✅ all stages started")

# 2.b no stage are still ongoing
for stage in test.stages:
    assert not stage.live(), f"❌ {stage.name()} is still live !!! Did you played it entirely ?"
print("✅ No stage is live")

# 2.c each stage have been terminated
for stage in test.stages:
    assert stage.over(), f"❌ {stage.name()} has not yet started !!! Did you forget to call stage.trophies() ?"
print("✅ All stage have finished")


# 3. Standing Output
for team in test.stages[-1].participants():
    assert test.standing()[team] == test.stages[-1].standing()[team], f"❌ {team} has an inconsistent final standing"
print("✅ Good Final Standing")
    
# 4. Game dataset
assert test.games() == [game for stage in test.stages for game in stage.games()], "❌ problem with the games output"
print("✅ clean games output")

✅ all teams in event
✅ all teams in first stage
✅ all stages started
✅ No stage is live
✅ All stage have finished
✅ Good Final Standing
✅ clean games output


## Invitations & Qualifications

In the calendar, the only garantee is that each team participates in the Play-ins of all 3 three splits of its region. After that everything is based on merit. We propose two approach to register teams to different events and stages. 

- Qualifications: within a Staged Event, teams move to the next stages based on their final placement, e.g "top8" goes to the main stage
- Invitations: Some teams do not participate in early stages and get directly invited to later stage

The GPR update rating based on event relevance. We define a naming convention for events that include all pieces of information necessary to tune the ratings update. 



In [3]:
from project.scheduler.calendar import year_schedule
from project.gpr.utils import EventInfos, get_event_infos, EVENT_NAMING, STAGED_EVENT_NAMING
from project.scene import Region, Split, Stage


# Create an event name based on event infos
infos = EventInfos(region=Region.LEC, split=Split.Summer, year=2025, stage='')
event = StagedEvent(name=str(infos), seeding=gt, solver=BetterWin(),
                    stages=[RoundRobin, SingleEliminationBracket, DoubleEliminationBracket],
                    stage_names=[Stage.PlayIns, Stage.MainStage, Stage.PlayOffs])

# infos versus name
print(infos, type(infos))
print(event.name())

# extracted infos from event name
event_infos = get_event_infos(event)
print(event_infos, type(infos))

LEC Summer 2025  <class 'project.gpr.utils.EventInfos'>
LEC Summer 2025 
LEC Summer 2025 Staged <class 'project.gpr.utils.EventInfos'>


#### TODO:
- support qualifications
- support invitations
- give proper name to event's stages

In [4]:
from project import StagedEventV2

# simulation parameters
playins_teams   = Player.create(nb=8)
mainstage_teams = Player.create(nb=4)
playoffs_teams  = Player.create(nb=4)
gt = BTRanking(name='GroundTruth', players=playins_teams+mainstage_teams+playoffs_teams)
solver = BetterWin()

# competition parameters
stages = [RoundRobin, SingleEliminationBracket, DoubleEliminationBracket]
names = [Stage.PlayIns, Stage.MainStage, Stage.PlayOffs]
nb_mainstage = 4
nb_playoffs = 4
invites = [playins_teams, mainstage_teams, playoffs_teams]
qualifications = [[], list(range(nb_mainstage)), list(range(nb_playoffs))]

# try your implementation
test = StagedEventV2(name='test', seeding=gt, solver=solver, stages=stages, stage_names=names)
test.registration(invited=invites, qualified=qualifications)
test.run()


# 1.a check all teams participated in the tournaments
assert set(test.participants()) == set(playins_teams+mainstage_teams+playoffs_teams), "❌ problem with the teams participating in the event"
print("✅ All teams in event")

# 1.b check all teams participated in the first stage
assert set(test.stages[0].participants()) == set(playins_teams), "❌ problem with the teams participating in the first stage"
print("✅ All teams in first stage")

# 2.a each stage have started
for stage in test.stages:
    assert stage.started(), f"❌ {stage.name()} has not yet started !!! Did you forget to call stage.start() ?"
print("✅ All stages started")

# 2.b no stage are still ongoing
for stage in test.stages:
    assert not stage.live(), f"❌ {stage.name()} is still live !!! Did you played it entirely ?"
print("✅ No stage is live")

# 2.c each stage have been terminated
for stage in test.stages:
    assert stage.over(), f"❌ {stage.name()} has not yet started !!! Did you forget to call stage.trophies() ?"
print("✅ All stage have finished")

# 3.a Standing with no tie
assert set(range(1, len(test.participants())+1)) == set([test.standing()[team] for team in test.participants()]), f"❌ Standing not covering proper ranks."
print("✅ All ranks in Standing")

# 3.b Standing coeherent with playoffs results
for team in test.stages[-1].participants():
    # placement at the end of the series of tournament
    event_placement = test.standing()[team]
    
    # placement in the last event of the series
    final_placement = test.stages[-1].standing()[team]

    # event_placement should be final_placement except in case two teams have the same final_placement
    if event_placement != final_placement:
        # the only other participant that has the event_placement team should have
        others = [t for t, p in test.standing().items() if p == event_placement]
        # other had the same final_placement
        for other in others:
            assert final_placement == test.stages[-1].standing()[other], f"❌ {team} has an inconsistent final standing"
print("✅ Coherent final Standing")
    
# 4. Game dataset
assert len(test.games()) == len([game for stage in test.stages for game in stage.games()]), "❌ problem with the games output"
print("✅ clean games output")

# 5. Qualifications system
assert len(test.stages[-1].participants()) == nb_playoffs + len(playoffs_teams), "❌ wrong number of teams in playoffs"
print("✅ proper qualifications")

✅ All teams in event
✅ All teams in first stage
✅ All stages started
✅ No stage is live
✅ All stage have finished
✅ All ranks in Standing
✅ Coherent final Standing
✅ clean games output
✅ proper qualifications


## Calendar

#### Materials:
- project/scheduler/calendar include an "EventInfos" dataclass to encapsulate tournament informations and automatically handle its name.
- project/scheduler/calendar also provides a year_schedule function that return an order list of event information